## Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.


### Pydantic

Pydantic models provide the richest feature set with field validation, description, and nested structures.

In [1]:
import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=ChatGroq(model="qwen/qwen3-32b")
model


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021934254DF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021934254D00>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    director:str=Field(description="The director of the movie")
    year:int=Field(description="The year the movie was released")
    genre:str=Field(description="The genre of the movie")
    rating:float=Field(description="The rating of the movie")

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021934254DF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021934254D00>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'genre': {'description': 'The genre of the movie', 'type': 'string'}, 'rating': {'description': 'The rating o

In [ ]:
model.invoke("Provide the details about the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. Inception is a 2010 movie directed by Christopher Nolan. The main actor is Leonardo DiCaprio, right? He plays the lead character, Dom Cobb. The genre is a mix of sci-fi and action, maybe some thriller elements. The plot involves dreams within dreams, something about entering people\'s minds to plant ideas. I remember there\'s a concept called "inception" which is like planting an idea without the person knowing. \n\nThe movie has some really cool visual effects, like the rotating hallway fight scene. The music is by Hans Zimmer, which gives it a really intense and emotional feel. The cast includes people like Joseph Gordon-Levitt, Ellen Page, Tom Hardy, and others. There\'s a lot of action set pieces, like the city folding in on itself or people falling through the sky. \n\nThe story is a bit complex. Cobb is a thief who steals secrets by infiltratin

In [5]:
model_with_structure.invoke("Provide the details about the movie Inception")

Movie(title='Inception', director='Christopher Nolan', year=2010, genre='Science Fiction', rating=8.8)

### Message output alongside Parsed Structure

In [6]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    director:str=Field(description="The director of the movie")
    year:int=Field(description="The year the movie was released")
    genre:str=Field(description="The genre of the movie")
    rating:float=Field(description="The rating of the movie")

model_with_structure = model.with_structured_output(Movie , include_raw=True)


response = model_with_structure.invoke("Provide the details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me see what I need to do here. The available tool is the Movie function, which requires specific parameters: title, director, year, genre, and rating. The user wants information on Inception, so I need to gather those details.\n\nFirst, I know the title is "Inception". The director is Christopher Nolan, right? The release year was 2010. The genre is probably science fiction or action. As for the rating, I think it\'s pretty high, maybe around 8.8 on IMDb. Let me double-check those facts to make sure I\'m not making a mistake. Yep, director is Nolan, year 2010, genre is sci-fi thriller, and the rating is indeed 8.8. So I should structure the function call with all these parameters. Make sure the JSON is correctly formatted with the required fields. No need to include any extra information beyond what\'s specified in the parameters. Alright, that shou

### Nested Structure

In [7]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genre:list[str]
    budget:float | None = Field(None, description="Budget in million USD")
    

In [8]:
model_with_structure = model.with_structured_output(MovieDetails)


response = model_with_structure.invoke("Provide the details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Bane'), Actor(name='Dileep Rao', role='Saito')], genre=['Action', 'Science Fiction', 'Heist'], budget=160.0)

### TypedDict

TypeDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [15]:
from typing_extensions import TypedDict, Annotated

In [22]:
class MovieDict(TypedDict):
    """A movie with Details"""
    title: Annotated[str,..., "The title of the movie"]
    director: Annotated[str,..., "The director of the movie"]
    year: Annotated[int,..., "The year the movie was released"]
    rating: Annotated[float,..., "The rating of the movie"]
    

In [23]:
model_with_typeddict = model.with_structured_output(MovieDict)
response = model_with_typeddict.invoke("Provide me the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [25]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genre:list[str]
    budget:float | None = Field(None, description="Budget in million USD")

model_with_typeddict = model.with_structured_output(MovieDetails)
response = model_with_typeddict.invoke("Provide me the details of the movie avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'genre': ['Action', 'Science Fiction', 'Superhero'],
 'title': 'Avengers',
 'year': 2012}

In [ ]:
model.profile

###  DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator.

In [27]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    # Auto-selects ProviderStrategy
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='1938e325-f623-433b-8e90-6940b2527a0c'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'f7mpceyd6', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 288, 'total_tokens': 321, 'completion_time': 0.063070785, 'completion_tokens_details': None, 'prompt_time': 0.015276161, 'prompt_tokens_details': None, 'queue_time': 0.158346519, 'total_time': 0.078346946}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cfa0f-16cd-7bb2-bfef-e8bbd3de9411-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'j

In [28]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [30]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]


{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [32]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    # Auto-selects ProviderStrategy
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]


ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')